In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from impulse import RJPTSampler, RJMCMCProductSpace
from impulse import model_visitation_stats, bayes_factor_from_chain

%load_ext autoreload
%autoreload 2

## RJPTSampler: Hybrid RJMCMC + NUTS + Parallel Tempering

This example demonstrates `RJPTSampler`, which combines three sampling
strategies in a single class:

1. **RJMCMC** (birth/death proposals) for trans-dimensional model selection
2. **NUTS** (gradient-based sampling) for efficient within-model exploration
3. **Parallel Tempering** for improved mixing across models and modes

Each iteration runs:
- A Metropolis-Hastings step (including RJ proposals) on all temperature chains
- A NUTS step on the active continuous parameters of each chain (tempered gradient)
- Periodic PT swaps between adjacent temperature chains

**Problem**: Given noisy time-series data, determine how many sinusoidal
components are present and estimate their parameters.

We compare three modes of the sampler:
1. MH-only (like `PTSampler.from_rjmcmc()`)
2. MH + NUTS (providing an analytical gradient)
3. The improvement in mixing and convergence from adding NUTS

### Generate synthetic data

Three sinusoids with varying amplitudes buried in Gaussian noise.
The sampler should recover the correct number of sources and their parameters.

In [ ]:
rng = np.random.default_rng(42)

N_pts = 300
t = np.linspace(0, 2 * np.pi, N_pts)
sigma = 1.0

# Three true sinusoids: [amplitude, frequency, phase]
true_sources = np.array([
    [2.5, 0.80, 1.2],
    [1.5, 1.60, 0.5],
    [0.8, 2.50, 2.0],
])
n_true = len(true_sources)

signal = np.zeros(N_pts)
for A, f, phi in true_sources:
    signal += A * np.sin(2 * np.pi * f * t + phi)

data = signal + sigma * rng.standard_normal(N_pts)

# Print SNR table
snr_values = true_sources[:, 0] * np.sqrt(N_pts / 2) / sigma
print(f'{"Source":>6s}  {"A":>5s}  {"f (Hz)":>6s}  {"phase":>5s}  {"SNR":>6s}')
print('-' * 40)
for i, ((A, f, phi), snr) in enumerate(zip(true_sources, snr_values)):
    print(f'  {i+1:>4d}  {A:5.2f}  {f:6.2f}  {phi:5.2f}  {snr:6.1f}')

plt.figure(figsize=(10, 4))
plt.plot(t, data, '.', alpha=0.3, ms=2, label='Noisy data')
plt.plot(t, signal, 'r-', lw=2, alpha=0.8, label=f'True signal ({n_true} sinusoids)')
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.legend()
plt.title(f'Synthetic data: {n_true} sinusoids + Gaussian noise')
plt.tight_layout()

### Define the model

The user provides:
1. A **likelihood** that uses the first `(nmodel+1) * num_params` active parameters
2. A **prior** that checks bounds on **all** source parameters (active + inactive)
3. A **draw function** that samples one source's parameters from the prior
4. (Optional) A **gradient function** for the likelihood on active parameters

The RJMCMC and NUTS machinery handles everything else.

In [ ]:
MAX_SOURCES = 8
NUM_PARAMS = 3  # (amplitude, frequency, phase) per source

# Prior bounds
A_MIN, A_MAX = 0.0, 5.0
F_MIN, F_MAX = 0.01, 4.0
PHI_MIN, PHI_MAX = 0.0, 2 * np.pi


def source_prior_draw(rng):
    """Draw one source's parameters from the prior."""
    return np.array([
        rng.uniform(A_MIN, A_MAX),
        rng.uniform(F_MIN, F_MAX),
        rng.uniform(PHI_MIN, PHI_MAX),
    ])


def log_prior(params):
    """Uniform prior on ALL source parameters (active + inactive)."""
    n = len(params) // NUM_PARAMS
    if n == 0:
        return 0.0
    p = params.reshape(n, NUM_PARAMS)
    if (np.any(p[:, 0] < A_MIN) or np.any(p[:, 0] > A_MAX)
        or np.any(p[:, 1] < F_MIN) or np.any(p[:, 1] > F_MAX)
        or np.any(p[:, 2] < PHI_MIN) or np.any(p[:, 2] > PHI_MAX)):
        return -np.inf
    return 0.0


def log_likelihood(params):
    """Gaussian likelihood for sinusoidal model (vectorized over sources)."""
    n = len(params) // NUM_PARAMS
    if n == 0:
        return -0.5 * np.sum((data / sigma) ** 2)
    p = params.reshape(n, NUM_PARAMS)
    arg = 2 * np.pi * np.outer(p[:, 1], t) + p[:, 2, None]
    model = np.sum(p[:, 0, None] * np.sin(arg), axis=0)
    return -0.5 * np.sum(((data - model) / sigma) ** 2)


def loglike_and_grad(active_params):
    """Log-likelihood and analytical gradient for active sources.

    Returns (loglike, grad) where grad has the same shape as active_params.
    The gradient is vectorized: no Python loops over sources or data points.
    """
    n = len(active_params) // NUM_PARAMS
    if n == 0:
        return -0.5 * np.sum((data / sigma) ** 2), np.array([])

    p = active_params.reshape(n, NUM_PARAMS)
    A, f, phi = p[:, 0], p[:, 1], p[:, 2]

    arg = 2 * np.pi * np.outer(f, t) + phi[:, None]     # (n, N_pts)
    sin_arg = np.sin(arg)
    cos_arg = np.cos(arg)

    model = np.sum(A[:, None] * sin_arg, axis=0)         # (N_pts,)
    residual = data - model
    loglike = -0.5 * np.sum((residual / sigma) ** 2)

    r_s2 = residual / sigma ** 2                          # (N_pts,)
    grad = np.empty_like(active_params)
    grad[0::3] = sin_arg @ r_s2                           # dL/dA
    grad[1::3] = (A[:, None] * (2 * np.pi * t[None, :]) * cos_arg) @ r_s2  # dL/df
    grad[2::3] = (A[:, None] * cos_arg) @ r_s2           # dL/dphi

    return loglike, grad

### Build the RJMCMC product space

The product space embeds all possible model configurations in a single
parameter vector. The model index (last parameter) determines which
sources are active.

In [ ]:
space = RJMCMCProductSpace(
    loglikelihood=log_likelihood,
    logprior=log_prior,
    num_sources=MAX_SOURCES,
    num_params=NUM_PARAMS,
    source_prior_draw=source_prior_draw,
)

print(f'Product space: {space.ndim} dimensions')
print(f'  {MAX_SOURCES} sources x {NUM_PARAMS} params/source + 1 model index')

---

## Run 1: MH-only (no NUTS)

First, use `RJPTSampler.from_rjmcmc()` **without** a gradient function.
This is equivalent to `PTSampler.from_rjmcmc()` — only adaptive MH
proposals (AM, SCAM, DE) plus the RJ birth/death proposals.

In [ ]:
sampler_mh = RJPTSampler.from_rjmcmc(
    space,
    ntemps=8,
    seed=42,
    outdir='./chains_rjpt_mh',
    save_freq=5000,
)

x0 = space.draw_initial_position(rng, nmodel=0)
print(f'Initial nmodel = {int(x0[-1])} ({int(x0[-1]) + 1} source)')
print(f'NUTS enabled: {sampler_mh.nuts_enabled}')

sampler_mh.sample(x0, num_iterations=50_000)

---

## Run 2: MH + NUTS

Now provide the `lnlike_grad` argument. The sampler will interleave
NUTS steps after each MH step on the active continuous parameters.
Hot chains use a smaller tree depth (`hot_chain_max_depth=4`) for efficiency.

In [ ]:
sampler_nuts = RJPTSampler.from_rjmcmc(
    space,
    lnlike_grad=loglike_and_grad,
    ntemps=8,
    seed=42,
    outdir='./chains_rjpt_nuts',
    save_freq=5000,
    max_tree_depth=8,
    hot_chain_max_depth=4,
)

x0 = space.draw_initial_position(rng, nmodel=0)
print(f'NUTS enabled: {sampler_nuts.nuts_enabled}')

sampler_nuts.sample(x0, num_iterations=15_000)

---

## Model Selection Results

Compare the posterior model probabilities from both runs.

In [ ]:
chains_mh = sampler_mh.load_chain()
chains_nuts = sampler_nuts.load_chain()

burn_mh = 15_000
burn_nuts = 3_000

probs_mh = space.model_posterior_probs(chains_mh['samples'][0], burn=burn_mh)
probs_nuts = space.model_posterior_probs(chains_nuts['samples'][0], burn=burn_nuts)

print('=== Model Selection ===')
print(f'True number of sinusoids: {n_true}')
print(f'Most probable model (MH-only):  {np.argmax(probs_mh) + 1} sinusoid(s)')
print(f'Most probable model (MH+NUTS):  {np.argmax(probs_nuts) + 1} sinusoid(s)')
print()
print(f'{"Sources":>8s}  {"MH-only":>8s}  {"MH+NUTS":>8s}')
print('-' * 30)
for k in range(MAX_SOURCES):
    if probs_mh[k] > 0.001 or probs_nuts[k] > 0.001:
        print(f'{k+1:>8d}  {probs_mh[k]:8.3f}  {probs_nuts[k]:8.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

model_range = np.arange(MAX_SOURCES) + 1

axes[0].bar(model_range, probs_mh, color='steelblue', edgecolor='navy')
axes[0].axvline(n_true, color='r', ls='--', lw=2, label=f'True = {n_true}')
axes[0].set_xlabel('Number of sinusoids')
axes[0].set_ylabel('Posterior probability')
axes[0].set_title(f'MH-only (50k iterations, burn={burn_mh})')
axes[0].legend()

axes[1].bar(model_range, probs_nuts, color='darkorange', edgecolor='saddlebrown')
axes[1].axvline(n_true, color='r', ls='--', lw=2, label=f'True = {n_true}')
axes[1].set_xlabel('Number of sinusoids')
axes[1].set_title(f'MH+NUTS (15k iterations, burn={burn_nuts})')
axes[1].legend()

plt.suptitle('Model posterior probabilities', fontsize=13, y=1.02)
plt.tight_layout()

### Trace comparison

Compare the model index traces and log-likelihood convergence.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))

# MH-only traces
cold_mh = chains_mh['samples'][0]
axes[0, 0].plot(cold_mh[:, -1], alpha=0.5, lw=0.5)
axes[0, 0].axhline(n_true - 1, color='r', ls='--', label=f'True nmodel = {n_true - 1}')
axes[0, 0].axvline(burn_mh, color='k', ls=':', alpha=0.5)
axes[0, 0].set_ylabel('nmodel')
axes[0, 0].set_title('MH-only: Model index')
axes[0, 0].legend(fontsize=8)

axes[1, 0].plot(chains_mh['lnlike'][0], alpha=0.5, lw=0.5)
axes[1, 0].axvline(burn_mh, color='k', ls=':', alpha=0.5)
axes[1, 0].set_ylabel('Log-likelihood')
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].set_title('MH-only: Log-likelihood')

# MH+NUTS traces
cold_nuts = chains_nuts['samples'][0]
axes[0, 1].plot(cold_nuts[:, -1], alpha=0.5, lw=0.5, color='darkorange')
axes[0, 1].axhline(n_true - 1, color='r', ls='--', label=f'True nmodel = {n_true - 1}')
axes[0, 1].axvline(burn_nuts, color='k', ls=':', alpha=0.5)
axes[0, 1].set_title('MH+NUTS: Model index')
axes[0, 1].legend(fontsize=8)

axes[1, 1].plot(chains_nuts['lnlike'][0], alpha=0.5, lw=0.5, color='darkorange')
axes[1, 1].axvline(burn_nuts, color='k', ls=':', alpha=0.5)
axes[1, 1].set_xlabel('Iteration')
axes[1, 1].set_title('MH+NUTS: Log-likelihood')

plt.suptitle('Convergence comparison', fontsize=13, y=1.02)
plt.tight_layout()

### NUTS diagnostics

The `get_diagnostics()` method summarises NUTS tree depth, divergences,
and parallel tempering swap rates.

In [ ]:
diag = sampler_nuts.get_diagnostics()

print('=== NUTS Diagnostics ===')
print(f'  Divergences:         {diag.get("num_divergent", "N/A")}')
print(f'  Max-depth hits:      {diag.get("num_max_depth", "N/A")}')
print(f'  Mean tree depth:     {diag.get("mean_tree_depth", "N/A"):.1f}')
print(f'  Mean accept prob:    {diag.get("mean_accept_prob", "N/A"):.3f}')
print(f'  Final step size:     {diag.get("final_step_size", "N/A"):.4f}')

print(f'\n=== PT Swap Acceptance Rates ===')
swap_rates = diag.get('pt_swap_accept', [])
temps = sampler_nuts.ptstate.ladder
for j, rate in enumerate(swap_rates):
    print(f'  T={temps[j]:.2f} <-> T={temps[j+1]:.2f}: {rate:.1%}')

In [ ]:
# NUTS diagnostics trace
if 'tree_depth' in chains_nuts:
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

    axes[0].plot(chains_nuts['tree_depth'], alpha=0.5, lw=0.5)
    axes[0].set_ylabel('Tree depth')
    mean_depth = np.mean(chains_nuts['tree_depth'][burn_nuts:])
    axes[0].set_title(f'Cold chain NUTS tree depth (mean = {mean_depth:.1f})')

    axes[1].plot(chains_nuts['step_size'], alpha=0.5, lw=0.5)
    axes[1].set_ylabel('Step size')
    axes[1].set_xlabel('Iteration')
    axes[1].set_title('Cold chain NUTS step size')

    plt.tight_layout()

### Parameter recovery

Examine the recovered source parameters from the MH+NUTS run.

In [ ]:
preferred = np.argmax(probs_nuts)
nmodel_samples = np.rint(cold_nuts[burn_nuts:, -1]).astype(int)
mask = nmodel_samples == preferred
preferred_samples = cold_nuts[burn_nuts:][mask]

print(f'=== Parameter recovery ({preferred + 1}-source model, '
      f'{mask.sum()} samples) ===')
print()

# Collect median parameters, sorted by frequency
recovered = []
for i in range(preferred + 1):
    A_med = np.median(preferred_samples[:, 3 * i])
    f_med = np.median(preferred_samples[:, 3 * i + 1])
    phi_med = np.median(preferred_samples[:, 3 * i + 2])
    recovered.append([A_med, f_med, phi_med])
recovered = sorted(recovered, key=lambda x: x[1])

true_sorted = sorted(true_sources.tolist(), key=lambda x: x[1])

print(f'{"":>5s}  {"--- Recovered ---":>26s}  |  {"--- True ---":>20s}')
print(f'{"#":>5s}  {"A":>7s} {"f":>7s} {"phi":>7s}  |  '
      f'{"A":>7s} {"f":>7s} {"phi":>7s}')
print('-' * 60)

for idx, (A_r, f_r, phi_r) in enumerate(recovered):
    # Match to closest true source by frequency
    dists = [abs(f_r - ts[1]) for ts in true_sorted]
    ci = int(np.argmin(dists))
    A_t, f_t, phi_t = true_sorted[ci]
    print(f'{idx+1:>5d}  {A_r:7.3f} {f_r:7.3f} {phi_r:7.3f}  |  '
          f'{A_t:7.3f} {f_t:7.3f} {phi_t:7.3f}')

### Best-fit model vs data

In [ ]:
cold_lnlike = chains_nuts['lnlike'][0]
best_idx = np.argmax(cold_lnlike[burn_nuts:]) + burn_nuts
best_params = cold_nuts[best_idx]
best_nmodel = int(np.rint(best_params[-1]))

best_model = np.zeros(N_pts)
for i in range(best_nmodel + 1):
    A = best_params[i * NUM_PARAMS]
    f = best_params[i * NUM_PARAMS + 1]
    phi = best_params[i * NUM_PARAMS + 2]
    best_model += A * np.sin(2 * np.pi * f * t + phi)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, data, '.', alpha=0.3, ms=2, label='Data')
axes[0].plot(t, signal, 'r-', lw=2, alpha=0.7, label=f'True ({n_true} sinusoids)')
axes[0].plot(t, best_model, 'k--', lw=1.5,
             label=f'Best fit ({best_nmodel + 1} sinusoids)')
axes[0].legend()
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Data and model comparison')

residual = data - best_model
axes[1].plot(t, residual, '.', alpha=0.3, ms=2)
axes[1].axhline(0, color='k', ls='-', alpha=0.3)
axes[1].axhline(sigma, color='r', ls='--', alpha=0.5, label=r'$\pm\sigma$')
axes[1].axhline(-sigma, color='r', ls='--', alpha=0.5)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Residual')
axes[1].set_title(f'Residuals (RMS = {np.std(residual):.3f})')
axes[1].legend()
plt.tight_layout()

### Visitation statistics and Bayes factors

In [ ]:
stats = model_visitation_stats(cold_nuts, num_models=MAX_SOURCES, burn=burn_nuts)

print('=== Model Visitation Statistics ===')
print()
for k in range(MAX_SOURCES):
    if stats['visit_counts'][k] > 0:
        pct = stats['visit_counts'][k] / stats['visit_counts'].sum() * 100
        dwell = stats['mean_dwell_times'][k]
        print(f'  {k+1:2d} sources: {int(stats["visit_counts"][k]):6d} visits '
              f'({pct:5.1f}%), mean dwell = {dwell:.1f} iters')

print(f'\n=== Bayes Factors (relative to {preferred+1}-source model) ===')
for k in range(MAX_SOURCES):
    if probs_nuts[k] > 0.001 and k != preferred:
        bf = probs_nuts[preferred] / probs_nuts[k]
        print(f'  B({preferred+1} vs {k+1}) = {bf:.1f}')

### Prior enforcement check

`RJMCMCProductSpace` enforces the prior on all source parameters
(including inactive ones) automatically.

In [ ]:
print('=== Prior enforcement check ===')
post_burn = cold_nuts[burn_nuts:]
all_ok = True
for i in range(MAX_SOURCES):
    A_vals = post_burn[:, i * NUM_PARAMS]
    f_vals = post_burn[:, i * NUM_PARAMS + 1]
    phi_vals = post_burn[:, i * NUM_PARAMS + 2]
    violations = (
        np.any(A_vals < A_MIN) or np.any(A_vals > A_MAX)
        or np.any(f_vals < F_MIN) or np.any(f_vals > F_MAX)
        or np.any(phi_vals < PHI_MIN) or np.any(phi_vals > PHI_MAX)
    )
    status = 'VIOLATION' if violations else 'OK'
    if violations:
        all_ok = False
    print(
        f'  Source {i + 1}:  '
        f'A in [{A_vals.min():.3f}, {A_vals.max():.3f}],  '
        f'f in [{f_vals.min():.3f}, {f_vals.max():.3f}],  '
        f'phi in [{phi_vals.min():.3f}, {phi_vals.max():.3f}]  '
        f'- {status}'
    )

nmodel_vals = np.rint(post_burn[:, -1]).astype(int)
valid = np.all((nmodel_vals >= 0) & (nmodel_vals < MAX_SOURCES))
if not valid:
    all_ok = False
print(f'  nmodel: all in {{0..{MAX_SOURCES - 1}}} = {valid}')
print(f'\nAll priors respected: {all_ok}')